# NYT Stage-1 Filter: Commodity-Market Relevance + High-Importance Flag

Stage 1 of the event-data pipeline (see `event_data_pipeline_plan_0417.md`). Runs on Kaggle (T4 16GB).

**Input**: NYT archive headlines (title + abstract + lead paragraph + date) fetched in `nyt_guardian_archive_fetch.ipynb`.

**Goal**: For each article, emit two flags using a Qwen-3 instruct model.
- `has_commodity_impact` — 1 unless the article has **absolutely zero** relevance to any commodity market. Drop only obviously irrelevant content: sports, entertainment, celebrity, arts, lifestyle, pure local crime.
- `high_importance` — 1 only for events expected to drive meaningful commodity-price movement (major geopolitical escalation, central-bank decisions, OPEC actions, large supply shocks, major macro surprises).

Bias is **recall over precision** — over-filtering destroys signal we can't recover.

**Output**: a parquet file with the two flags appended. Survivors feed Stage 2 (expand + verify).

### Model
`Qwen/Qwen3-14B` at 4-bit via bitsandbytes (~8GB). Largest dense Qwen3 that fits comfortably on a single T4 with room for KV cache. Uses non-thinking mode for speed.

### Output trick
Instead of JSON, the model emits two digits (e.g. `"10"` = has_impact=1, high_importance=0). Cuts generation from ~40 tokens to 3 — order-of-magnitude throughput gain at no cost to parsing robustness.

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch pandas pyarrow

In [ ]:
import re
import json
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_token = user_secrets.get_secret("HF_TOKEN")
login(token=HF_token)

MODEL_ID = "Qwen/Qwen3-14B"
# Alternatives if 14B runs slow or OOMs:
#   Qwen/Qwen3-8B                 (smaller, faster, still strong)
#   Qwen/Qwen3-4B-Instruct-2507   (fastest, used in existing GDELT filter)

INPUT_PARQUET = "/kaggle/input/datasets/zygong1994/finance-world-model-dataset/nyt_2015_2016.parquet"
OUTPUT_PARQUET = "nyt_2015_2016_filtered.parquet"
BATCH_SIZE = 8
MAX_LEAD_CHARS = 600  # truncate lead paragraph to keep prompts compact

## 1. Load NYT articles

In [ ]:
df = pd.read_parquet(INPUT_PARQUET)
print(f"Loaded {len(df)} NYT articles")
print(f"Date range: {df['date'].min()} -> {df['date'].max()}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

## 2. Load Qwen3-14B (4-bit)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print(f"Loaded {MODEL_ID} on {model.device}")
print(f"Peak memory so far: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## 3. Prompt + parse

Two digits: `XY` where `X` = `has_commodity_impact`, `Y` = `high_importance`.

Valid outputs: `00`, `10`, `11`. `01` is semantically invalid (high_importance requires some impact) and will be remapped to `11` if it ever appears.

In [ ]:
SYSTEM_PROMPT = (
    "You are a commodity-market analyst filtering news for a volatility model covering oil, "
    "natural gas, gold, silver, copper, agricultural commodities, and related markets.\n\n"
    "For each article, output exactly TWO digits, no other text:\n"
    "  First digit  = has_commodity_impact (1 = any plausible impact, 0 = zero impact)\n"
    "  Second digit = high_importance (1 = likely major price mover, 0 = not major)\n\n"
    "Rules for has_commodity_impact:\n"
    "- Mark 0 ONLY for clearly irrelevant content: sports, entertainment, celebrity news, "
    "arts/culture reviews, lifestyle, obituaries of non-economic figures, routine local "
    "crime with no economic angle.\n"
    "- Mark 1 for ANY plausible commodity linkage: geopolitics, energy policy, monetary "
    "policy, trade policy, corporate news from major producers/consumers, weather/disaster "
    "events, sanctions, supply-chain disruptions, inflation/labor data, regulation, fiscal "
    "policy, major political instability.\n"
    "- When uncertain, output 1. Err strongly on the side of keeping.\n\n"
    "Rules for high_importance:\n"
    "- Mark 1 only for events with clear potential to drive meaningful commodity price "
    "movement: OPEC decisions, central-bank rate actions, major geopolitical escalation "
    "(war, major sanctions, regime change in producer countries), large supply disruptions "
    "(refinery outages, pipeline attacks), surprise macro data releases, major fiscal "
    "announcements.\n"
    "- Mark 0 for routine business news, small-scale geopolitical friction, minor political "
    "noise, opinion pieces.\n"
    "- high_importance=1 requires has_commodity_impact=1.\n\n"
    "Valid outputs: 00, 10, 11. Output ONLY the two digits."
)

def build_article_block(row: dict) -> str:
    date = row.get("date", "")
    headline = row.get("headline", "") or ""
    abstract = row.get("abstract", "") or ""
    lead = (row.get("lead_paragraph", "") or "")[:MAX_LEAD_CHARS]
    return (
        f"Date: {date}\n"
        f"Headline: {headline}\n"
        f"Abstract: {abstract}\n"
        f"Lead: {lead}"
    )

def build_prompt(row: dict) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_article_block(row)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # Qwen3: disable thinking for speed
    )

# Preview
print(build_prompt(df.iloc[1].to_dict()))

In [ ]:
_DIGIT_RE = re.compile(r"([01])\s*([01])")

def parse_two_digits(text: str) -> tuple[int, int]:
    """Return (has_commodity_impact, high_importance). Defaults to (1, 0) if unparseable
    so we keep-when-unsure and don't over-flag as high-importance."""
    m = _DIGIT_RE.search(text)
    if not m:
        return 1, 0
    a, b = int(m.group(1)), int(m.group(2))
    # enforce semantic: high_importance requires has_impact
    if b == 1 and a == 0:
        a = 1
    return a, b

# Sanity checks
assert parse_two_digits("10") == (1, 0)
assert parse_two_digits("00") == (0, 0)
assert parse_two_digits("11") == (1, 1)
assert parse_two_digits("1 1") == (1, 1)
assert parse_two_digits("01") == (1, 1)  # invalid fixed
assert parse_two_digits("garbage") == (1, 0)
print("parse tests passed")

## 4. Batched classification

In [ ]:
@torch.no_grad()
def classify_batch(rows: list[dict]) -> list[tuple[int, int]]:
    prompts = [build_prompt(r) for r in rows]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        padding_side="left",
        truncation=True,
        max_length=1536,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    results = []
    for i, out in enumerate(outputs):
        input_len = inputs["input_ids"][i].shape[0]
        generated = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        results.append(parse_two_digits(generated))
    return results

In [ ]:
# Smoke test on a handful of articles
sample_rows = df.head(8).to_dict("records")
sample_results = classify_batch(sample_rows)
for row, (a, b) in zip(sample_rows, sample_results):
    print(f"[{a}{b}] {row['headline'][:100]}")

## 5. Run over the full dataset

In [ ]:
rows = df.to_dict("records")
has_impact_flags: list[int] = []
high_importance_flags: list[int] = []

start_time = time.time()
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i : i + BATCH_SIZE]
    results = classify_batch(batch)
    for a, b in results:
        has_impact_flags.append(a)
        high_importance_flags.append(b)

    if (i // BATCH_SIZE) % 10 == 0:
        elapsed = time.time() - start_time
        rate = len(has_impact_flags) / max(elapsed, 1e-6)
        eta = (len(rows) - len(has_impact_flags)) / max(rate, 1e-6)
        print(
            f"{len(has_impact_flags):5d}/{len(rows)}  "
            f"({len(has_impact_flags)/len(rows):.1%})  "
            f"{rate:.1f} art/s  ETA {eta/60:.1f} min"
        )

print(f"Done. Classified {len(has_impact_flags)} articles in {(time.time()-start_time)/60:.1f} min")

In [ ]:
df["has_commodity_impact"] = has_impact_flags
df["high_importance"] = high_importance_flags

n_total = len(df)
n_kept = int(df["has_commodity_impact"].sum())
n_high = int(df["high_importance"].sum())
print(f"Total:           {n_total}")
print(f"has_impact=1:    {n_kept:5d}  ({n_kept/n_total:.1%})")
print(f"high_importance: {n_high:5d}  ({n_high/n_total:.1%})")

In [ ]:
# Spot-check: what got dropped, and what got flagged high-importance
print("=== DROPPED (has_impact=0) — first 15 ===")
for _, r in df[df["has_commodity_impact"] == 0].head(15).iterrows():
    print(f"  [{r['section']:12s}] {r['headline'][:110]}")

print("\n=== HIGH IMPORTANCE — first 15 ===")
for _, r in df[df["high_importance"] == 1].head(15).iterrows():
    print(f"  [{r['section']:12s}] {r['headline'][:110]}")

In [ ]:
df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved {OUTPUT_PARQUET} ({Path(OUTPUT_PARQUET).stat().st_size/1024:.1f} KB)")